# 🎵 NINE-Music — Treinador GPU (Colab + git clone)

**1)** *Ambiente de execução → Alterar tipo de ambiente → GPU (T4).*
**2)** Rode as células em ordem. O código vem direto do GitHub (`Vtgamer998/nine-music`).

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução > Alterar tipo.'

## 1. Clonar o repositório e achar a pasta `nine_upgrade`

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/Vtgamer998/nine-music.git"
BRANCH = "main"
os.chdir("/content")
subprocess.run("rm -rf /content/nine-music", shell=True)
r = subprocess.run(f"git clone --depth 1 -b {BRANCH} {REPO} /content/nine-music", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)

found = subprocess.run("find /content/nine-music -type d -name nine_upgrade", shell=True, capture_output=True, text=True).stdout.split()
if not found:
    print("\n❌ A pasta nine_upgrade NAO esta no repositorio.")
    print("   Faca o commit/push da pasta 'nine_upgrade' (com __init__.py, model.py, train.py...) e rode de novo.")
    print("   Conteudo atual do repo:"); print(subprocess.run("ls /content/nine-music", shell=True, capture_output=True, text=True).stdout)
    raise SystemExit
ROOT = os.path.dirname(found[0])   # pasta que CONTEM nine_upgrade
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("✅ nine_upgrade encontrado em:", found[0]); print("pasta de trabalho:", ROOT)
print(os.listdir(found[0]))

## 2. Dependências (datasets + MIDI/soundfont)

In [ ]:
!pip install -q datasets
!apt-get install -y -q abcmidi fluidsynth fluid-soundfont-gm > /dev/null 2>&1
!python tests/test_upgrade.py 2>/dev/null || echo 'tests/test_upgrade.py nao esta no repo (ok, so opcional)'

## 3. Dados
`irishman` = dataset grande (HF). **Confira a licença** antes de distribuir o modelo. Só Nottingham: use `--sources nottingham`.

In [ ]:
!python -m nine_upgrade.prepare_data --out data --sources nottingham irishman --max_irishman 60000

## 4. Treino
Presets: `small` (~10M) · `base` (~25M, recomendado na T4) · `large` (~85M).
Se faltar memória: `--batch 16 --grad_accum 2`. Se o Colab cair: rode com `--resume`.

In [ ]:
PRESET = "base"      # small | base | large
ITERS  = 8000
BATCH  = 32
OUT    = f"runs/{PRESET}"
!python -m nine_upgrade.train --data data --out {OUT} --preset {PRESET} --iters {ITERS} --batch {BATCH}

### (opcional) Retomar treino interrompido

In [ ]:
# !python -m nine_upgrade.train --data data --out {OUT} --preset {PRESET} --iters {ITERS} --batch {BATCH} --resume

## 5. Gerar tunes (só as com compassos válidos) + áudio

In [ ]:
STYLE, KEY, NUM = "jig", "D", 3     # jig | reel | waltz | polka | march | slipjig | slide
!python -m nine_upgrade.generate --ckpt {OUT}/best.pt --tok {OUT}/tokenizer.json --style {STYLE} --key {KEY} --num {NUM} --out samples

In [ ]:
import glob, IPython.display as ipd
for w in sorted(glob.glob('samples/*.wav')):
    print(w); print(open(w.replace('.wav','.abc')).read()); ipd.display(ipd.Audio(w))

## 6. Salvar o modelo
Prefira o Google Drive (o Colab apaga tudo quando a sessão expira).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nine-music && cp -r {OUT} /content/drive/MyDrive/nine-music/
print("salvo no Drive: MyDrive/nine-music/" + OUT.split("/")[-1])

In [ ]:
from google.colab import files
files.download(f'{OUT}/best.pt'); files.download(f'{OUT}/tokenizer.json')